# CLARIN fragment finder — interactive

Identify short overlap-dense fragments in `~/datasets/clarin_all_2speakers/`
recordings so we can hand-transcribe a manageable subset and run the pipeline +
eval module against them.

The algorithm lives in `scripts/clarin_fragment_finder.py` (importable + CLI-callable).
This notebook calls it on one recording at a time and lets you inspect every
candidate fragment visually + by ear. Edit the parameter dict at the top of the
**find** cell and re-run to iterate.

Once you're happy with a parameter set, ping me and I'll batch-extract the fragments
into `~/datasets/clarin_all_2speaker_fragments/`.

In [ ]:
from IPython.display import HTML, display
display(HTML("<style>audio{width:100% !important;}</style>"))  # szersze odtwarzacze audio

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if Path('scripts').is_dir() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, Audio

import soundfile as sf

from scripts.clarin_fragment_finder import (
    FragmentParams, Fragment, Turn,
    load_diarization, overlap_intervals, speech_intervals, silence_gaps,
    find_fragments,
)

CLARIN_ROOT = Path('~/datasets/clarin_all_2speakers').expanduser()
DIAR_DIR    = CLARIN_ROOT / 'diarization'
AUDIO_DIR   = CLARIN_ROOT / 'clarin_download'

pd.set_option('display.float_format', lambda x: f'{x:.2f}')
pd.set_option('display.max_rows', 100)
print(f'diarizations available: {len(list(DIAR_DIR.glob("*.json")))}')
print(f'audio files available:  {len(list(AUDIO_DIR.glob("*.wav")))}')

## Plotting helpers

One helper that draws the per-speaker diarization timeline with overlap regions
highlighted in red, plus an optional list of `Fragment` rectangles overlaid.

In [ ]:
def plot_diarization(turns, total_dur, overlaps, fragments=None,
                     xlim=None, title=None, figsize=(14, 2.8)):
    """Per-speaker bars + overlap highlights. `fragments` is an optional
    list of `Fragment`s drawn as cyan boxes spanning the y-axis."""
    speakers = sorted({t.speaker for t in turns})
    if not speakers:
        print('no speakers in diarization')
        return
    fig, ax = plt.subplots(figsize=figsize)
    y_for = {spk: i for i, spk in enumerate(speakers)}
    for t in turns:
        ax.add_patch(Rectangle(
            (t.start, y_for[t.speaker] - 0.3), t.end - t.start, 0.6,
            facecolor='C0', edgecolor='none', alpha=0.7,
        ))
    for os_, oe in overlaps:
        ax.axvspan(os_, oe, color='red', alpha=0.15)
    if fragments:
        for f in fragments:
            ax.add_patch(Rectangle(
                (f.start, -0.7), f.end - f.start, len(speakers) - 1 + 1.4,
                facecolor='cyan', edgecolor='black', alpha=0.18,
                linewidth=0.8,
            ))
    ax.set_yticks(range(len(speakers)))
    ax.set_yticklabels(speakers)
    ax.set_xlim(xlim or (0, total_dur))
    ax.set_ylim(-0.8, len(speakers) - 1 + 0.8)
    ax.set_xlabel('time (s)')
    if title:
        ax.set_title(title)
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

## Pick a recording

Change `RECORDING_ID` below. By default we pick the first one alphabetically that
has both diarization and audio.

In [ ]:
# --- Knob ---
RECORDING_ID = None   # set to e.g. '005cba37' to pick one; None = auto-pick first

available = sorted([p.stem for p in DIAR_DIR.glob('*.json')
                    if (AUDIO_DIR / f'{p.stem}.wav').exists()])
print(f'{len(available)} recording(s) have both diarization + audio')
print(f'first 10: {available[:10]}')

if RECORDING_ID is None:
    RECORDING_ID = available[0]
diar_path  = DIAR_DIR  / f'{RECORDING_ID}.json'
audio_path = AUDIO_DIR / f'{RECORDING_ID}.wav'
assert diar_path.exists(),  f'missing: {diar_path}'
assert audio_path.exists(), f'missing: {audio_path}'

turns, total_dur = load_diarization(diar_path)
overlaps_all     = overlap_intervals(turns)
speech_all       = speech_intervals(turns)

n_overlap = len(overlaps_all)
total_overlap_s = sum(e - s for s, e in overlaps_all)
speakers = sorted({t.speaker for t in turns})
print(f'\n[{RECORDING_ID}]  {total_dur:.1f}s total ({total_dur/60:.1f} min)')
print(f'  turns:    {len(turns)}, speakers: {speakers}')
print(f'  overlaps: {n_overlap} intervals, {total_overlap_s:.1f}s ({total_overlap_s/total_dur*100:.1f}% of recording)')

## Full diarization timeline

Per-speaker activity (blue bars) + pyannote-detected overlap regions (red shading).
Long red runs are good — that's where the separator gets to work.

In [ ]:
plot_diarization(turns, total_dur, overlaps_all,
                 title=f'{RECORDING_ID} — full diarization (red = overlaps)')

## Find candidate fragments

Edit `params` to iterate. Defaults:

| knob | default | what it does |
| --- | --- | --- |
| `target_length_s` | 90 | the length we aim for; boundary snap may stretch ±5s |
| `min_length_s` / `max_length_s` | 30 / 150 | hard envelope (0.5–2.5 min) |
| `stride_s` | 5 | sliding-window stride; smaller = denser candidates, slower |
| `silence_snap_min_s` | 0.5 | min silence gap counted as a boundary |
| `silence_snap_search_s` | 5.0 | how far to look for silence when snapping |
| `min_overlap_s` | 3.0 | min total overlap inside a fragment to keep it |
| `min_speaker_balance` | 0.15 | min share for the weaker speaker (0=one-sided, 0.5=equal) |
| `min_speech_density` | 0.40 | min fraction of fragment that contains speech |
| `nms_iou_threshold` | 0.30 | lower = more fragments per recording (less de-duped) |

In [ ]:
params = FragmentParams(
    target_length_s=90.0,
    min_length_s=30.0,
    max_length_s=150.0,
    stride_s=5.0,
    silence_snap_min_s=0.5,
    silence_snap_search_s=5.0,
    min_overlap_s=3.0,
    min_speaker_balance=0.15,
    min_speech_density=0.40,
    nms_iou_threshold=0.30,
)

fragments = find_fragments(turns, total_dur, params)
print(f'found {len(fragments)} fragment(s)')

if fragments:
    df = pd.DataFrame([f.summary_dict() for f in fragments])
    display(df)
else:
    print('try loosening min_overlap_s or min_speaker_balance')

## Candidate fragments overlaid on the diarization

Cyan rectangles show each fragment's full span. If they're clustering on one side
of the recording, you're probably anchored on a single overlap-heavy region; nudge
`nms_iou_threshold` lower to keep nearby alternates, or check the full timeline
for missed regions.

In [ ]:
plot_diarization(turns, total_dur, overlaps_all, fragments=fragments,
                 title=f'{RECORDING_ID} — {len(fragments)} fragment(s)')

## Inspect each fragment (zoomed timeline + audio)

Per fragment: a zoomed-in diarization slice + an audio player so you can listen to
exactly what the pipeline would receive. Adjust `MAX_INSPECT` if there are too many.

In [ ]:
MAX_INSPECT = 8
PAD_S = 5.0   # context shown on either side of each fragment

# Load audio once (lazy: only if there are fragments to listen to)
audio_np = None
audio_sr = None
if fragments:
    audio_np, audio_sr = sf.read(str(audio_path), dtype='float32')
    if audio_np.ndim > 1:
        audio_np = audio_np.mean(axis=1)
    print(f'loaded audio: {len(audio_np)/audio_sr:.1f}s @ {audio_sr} Hz')

for i, f in enumerate(fragments[:MAX_INSPECT]):
    print()
    print(f'--- fragment {i}  [{f.start:.1f}, {f.end:.1f}]  '
          f'dur={f.duration:.1f}s  overlap={f.overlap_s:.1f}s  '
          f'balance={f.speaker_balance:.2f}  speech={f.speech_density*100:.0f}% ---')
    plot_diarization(
        turns, total_dur, overlaps_all,
        fragments=[f],          # cyan box marks the actual fragment
        xlim=(max(0, f.start - PAD_S), min(total_dur, f.end + PAD_S)),
        title=f'fragment {i}: [{f.start:.1f} \u2192 {f.end:.1f}] ({f.duration:.1f}s)  '
              f'\u2014 cyan box = fragment, \u00b1{PAD_S:.0f}s context shown either side',
        figsize=(12, 2.0),
    )
    if audio_np is not None:
        lo = int(f.start * audio_sr)
        hi = int(f.end * audio_sr)
        display(Audio(audio_np[lo:hi], rate=audio_sr))

## Batch view — how do these params look across every recording?

Walks every diarization JSON with the current `params` and reports per-recording
fragment counts + total fragment minutes. This is the **fast** check before committing
to a parameter set for batch extraction. Audio not loaded.

In [ ]:
rows = []
for p in sorted(DIAR_DIR.glob('*.json')):
    rid = p.stem
    turns_, total_ = load_diarization(p)
    frags = find_fragments(turns_, total_, params)
    rows.append({
        'recording': rid,
        'recording_min': round(total_ / 60, 1),
        'n_turns': len(turns_),
        'n_fragments': len(frags),
        'total_fragment_min': round(sum(f.duration for f in frags) / 60, 1),
        'total_overlap_in_fragments_s': round(sum(f.overlap_s for f in frags), 1),
    })

batch_df = pd.DataFrame(rows)
print(f'recordings with ≥ 1 fragment: {(batch_df["n_fragments"] > 0).sum()} / {len(batch_df)}')
print(f'total fragment minutes across the dataset: '
      f'{batch_df["total_fragment_min"].sum():.1f} min '
      f'({batch_df["total_fragment_min"].sum() / 60:.1f} hours)')
print(f'recordings with 0 fragments (no useful overlap or filters too tight): '
      f'{(batch_df["n_fragments"] == 0).sum()}')
print()
print('per-recording (sorted by n_fragments desc):')
display(batch_df.sort_values('n_fragments', ascending=False).head(30))

## When you're ready

Once the params look right (the table above has a reasonable spread of fragments,
the inspected ones sound like things worth transcribing), copy the `params` block
and paste it back to me. I'll wire it into a batch script that writes the audio
fragments + a manifest into `~/datasets/clarin_all_2speaker_fragments/`.